In [1]:
!pip install -U scikit-learn==1.5.0
!pip install -U imbalanced-learn==0.13.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 52.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 7.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sklearn
import imblearn

print("scikit-learn version:", sklearn.__version__)
print("imblearn version:", imblearn.__version__)

scikit-learn version: 1.5.0
imblearn version: 0.13.0


In [4]:
!pip install polars[numpy,pandas,pyarrow] --no-index --find-links=file:///kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/polars_pkg

Looking in links: file:///kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/polars_pkg
Processing /kaggle/input/polars-and-duckdb/kaggle/working/mysitepackages/polars_pkg/polars-0.20.16-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl


In [5]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import glob
import os
import polars as pl
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import pickle
import ctypes
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

E0000 00:00:1745167071.910761      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


In [6]:
# detect TPUs
tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
tf.tpu.experimental.initialize_tpu_system(tpu)
tpu_strategy = tf.distribute.TPUStrategy(tpu)

print("Number of accelerators: ", tpu_strategy.num_replicas_in_sync)

INFO:tensorflow:Deallocate tpu buffers before initializing tpu system.
INFO:tensorflow:Initializing the TPU system: local


I0000 00:00:1745167156.086092      10 service.cc:148] XLA service 0x5d077c978e40 initialized for platform TPU (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745167156.086140      10 service.cc:156]   StreamExecutor device (0): TPU, 2a886c8
I0000 00:00:1745167156.086144      10 service.cc:156]   StreamExecutor device (1): TPU, 2a886c8
I0000 00:00:1745167156.086147      10 service.cc:156]   StreamExecutor device (2): TPU, 2a886c8
I0000 00:00:1745167156.086150      10 service.cc:156]   StreamExecutor device (3): TPU, 2a886c8
I0000 00:00:1745167156.086153      10 service.cc:156]   StreamExecutor device (4): TPU, 2a886c8
I0000 00:00:1745167156.086156      10 service.cc:156]   StreamExecutor device (5): TPU, 2a886c8
I0000 00:00:1745167156.086158      10 service.cc:156]   StreamExecutor device (6): TPU, 2a886c8
I0000 00:00:1745167156.086161      10 service.cc:156]   StreamExecutor device (7): TPU, 2a886c8


INFO:tensorflow:Finished initializing TPU system.
INFO:tensorflow:Found TPU system:
INFO:tensorflow:*** Num TPU Cores: 8
INFO:tensorflow:*** Num TPU Workers: 1
INFO:tensorflow:*** Num TPU Cores Per Worker: 8
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:CPU:0, CPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:0, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:1, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:2, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:3, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:4, TPU, 0, 0)
INFO:tensorflow:*** Available Device: _DeviceAttributes(/job:localhost/replica:0/task:0/device:TPU:5, TPU, 0, 0)
I

In [7]:
feature_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/extracted_train_feat_five_sec_v5_2025'
label_file_path = '/kaggle/input/birdclef-2025-features-labels-v1/labels_five_sec_v5_2025'

missing_classes_feature_file_path = '/kaggle/input/missing-classes-features-labels/late_inclusion_features_2025'
missing_classes_label_file_path = '/kaggle/input/missing-classes-features-labels/late_inclusion_labels_2025'

with open(feature_file_path, "rb") as file:
    pickled_extracted_features_five_sec = pickle.load(file)
    
with open(label_file_path, "rb") as file:
    labels_five_sec = pickle.load(file)

with open(missing_classes_feature_file_path, "rb") as file:
    pickled_missing_classes_features_five_sec = pickle.load(file)
    
with open(missing_classes_label_file_path, "rb") as file:
    labels_missing_classes_five_sec = pickle.load(file)

In [10]:
print("Previous labels shape:", labels_five_sec.shape)
print("Previous labels dtype:", labels_five_sec.dtype)
print("Previous labels sample:", labels_five_sec[:5])  # Show first 5 elements

print("\nNew labels shape:", labels_missing_classes_five_sec.shape)
print("New labels dtype:", labels_missing_classes_five_sec.dtype)
print("New labels sample:", labels_missing_classes_five_sec[:5])

Previous labels shape: (180142,)
Previous labels dtype: int64
Previous labels sample: [110 110 110 110 110]

New labels shape: (620,)
New labels dtype: int64
New labels sample: [17 17 17 17 17]


In [11]:
x_five_sec = np.vstack([pickled_extracted_features_five_sec, pickled_missing_classes_features_five_sec])
y_five_sec = np.concatenate([labels_five_sec, labels_missing_classes_five_sec], axis=0)

print("Combined features shape:", x_five_sec.shape)
print("Combined labels shape:", y_five_sec.shape)

Combined features shape: (180762, 40)
Combined labels shape: (180762,)


In [13]:
ros = RandomOverSampler(random_state=42, sampling_strategy='minority')
features_resampled, labels_resampled = ros.fit_resample(x_five_sec, y_five_sec)

print("Resampled features shape:", features_resampled.shape)
print("Resampled labels shape:", labels_resampled.shape)

Resampled features shape: (186377, 40)
Resampled labels shape: (186377,)


In [14]:
shuffle_idx = np.random.permutation(len(features_resampled))
X_combined = features_resampled[shuffle_idx]
y_combined = labels_resampled[shuffle_idx]

In [38]:
def create_model(input_shape, num_classes):
    model = models.Sequential([
        # Input layer with batch normalization
        layers.Input(shape=input_shape),
        layers.BatchNormalization(),
        
        # First dense block
        layers.Dense(512, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),
        
        # Second dense block
        layers.Dense(256, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        
        # Third dense block
        layers.Dense(128, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

In [36]:
def prepare_dataset(X, y, batch_size=32):
    """
    Convert numpy arrays to tf.data.Dataset with batching and prefetching
    """
    min_label = np.min(y)
    max_label = np.max(y)
    unique_labels = np.unique(y)
    print(f"Label range: {min_label} to {max_label}")
    print(f"Number of unique labels: {len(unique_labels)}")
    #print(f"Unique labels: {unique_labels}")
    
    # Robust normalization using min-max scaling instead of standard scaling
    X_min = np.min(X, axis=0)
    X_max = np.max(X, axis=0)
    X_normalized = (X - X_min) / (X_max - X_min + 1e-8)
    
    # Double check normalization
    print("After normalization:")
    print("Min:", np.min(X_normalized))
    print("Max:", np.max(X_normalized))
    print("Mean:", np.mean(X_normalized))
    print("Std:", np.std(X_normalized))
    
    # Convert labels to categorical (one-hot encoding)
    num_classes = max_label + 1
    print(f"Setting num_classes to: {num_classes}")
    
    y_categorical = tf.keras.utils.to_categorical(y, num_classes=num_classes)
    
    # Create tf.data.Dataset
    dataset = tf.data.Dataset.from_tensor_slices((X_normalized, y_categorical))
    #dataset = tf.data.Dataset.from_tensor_slices((X_normalized, y))
    dataset = dataset.cache()  # Cache the data in memory
    dataset = dataset.shuffle(buffer_size=len(X))  # Shuffle the entire dataset
    dataset = dataset.batch(batch_size)  # Batch the data
    dataset = dataset.prefetch(tf.data.AUTOTUNE)  # Prefetch next batch
    
    return dataset

In [37]:
X_train, X_val, y_train, y_val = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42)

# Prepare datasets
train_dataset = prepare_dataset(X_train, y_train, batch_size=64)
val_dataset = prepare_dataset(X_val, y_val, batch_size=64)

Label range: 0 to 205
Number of unique labels: 205
After normalization:
Min: 0.0
Max: 1.0
Mean: 0.4960604
Std: 0.08911721
Setting num_classes to: 206
Label range: 0 to 205
Number of unique labels: 200
After normalization:
Min: 0.0
Max: 1.0
Mean: 0.49804592
Std: 0.096977696
Setting num_classes to: 206


In [22]:
input_shape = (X_train.shape[1],)  # Will be (40,)
num_classes = 206

In [32]:
with tpu_strategy.scope():
    model = create_model(input_shape, num_classes)

    initial_learning_rate = 1e-5
        
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=1000,
        decay_rate=1.1,  # Slight increase during warm-up
        staircase=True
    )
    
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=lr_schedule,
        clipnorm=1.0,
        epsilon=1e-7,
        beta_1=0.9,
        beta_2=0.999
    )
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

In [24]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6
    )
]

In [35]:
# Quick data inspection
print("Feature statistics:")
print("Shape:", X_combined.shape)
print("Any NaN:", np.isnan(X_combined).any())
print("Any Inf:", np.isinf(X_combined).any())
print("Min value:", np.min(X_combined))
print("Max value:", np.max(X_combined))
print("Mean:", np.mean(X_combined))
print("Std:", np.std(X_combined))

Feature statistics:
Shape: (186377, 40)
Any NaN: False
Any Inf: False
Min value: -1029.1322
Max value: 307.16528
Mean: -7.867528
Std: 62.314613


In [33]:
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    callbacks=callbacks
)

Epoch 1/50


I0000 00:00:1745168939.093005      10 encapsulate_tpu_computations_pass.cc:266] Subgraph fingerprint:13325865783830652614
I0000 00:00:1745168939.824207     842 tpu_compilation_cache_interface.cc:442] TPU host compilation cache miss: cache_key(1929766531465674985), session_name()
I0000 00:00:1745168942.774042     842 tpu_compile_op_common.cc:245] Compilation of 1929766531465674985 with session name  took 2.949771182s and succeeded
I0000 00:00:1745168942.793189     842 tpu_compilation_cache_interface.cc:476] TPU host compilation cache: compilation complete for cache_key(1929766531465674985), session_name(), subgraph_key(std::string(property.function_name) = "cluster_one_step_on_data_13325865783830652614", property.function_library_fingerprint = 6837765850498572029, property.mlir_module_fingerprint = 0, property.num_replicas = 8, topology.chip_bounds().x = 2, topology.chip_bounds().y = 2, topology.chip_bounds().z = 1, topology.wrap().x = false, topology.wrap().y = false, topology.wrap().z

2329/2330 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: nan - loss: nan

I0000 00:00:1745168965.033071     826 tpu_compilation_cache_interface.cc:442] TPU host compilation cache miss: cache_key(4550261933820635953), session_name()
I0000 00:00:1745168968.212141     826 tpu_compile_op_common.cc:245] Compilation of 4550261933820635953 with session name  took 3.179033469s and succeeded
I0000 00:00:1745168968.235308     826 tpu_compilation_cache_interface.cc:476] TPU host compilation cache: compilation complete for cache_key(4550261933820635953), session_name(), subgraph_key(std::string(property.function_name) = "cluster_one_step_on_data_13325865783830652614", property.function_library_fingerprint = 6837765850498572029, property.mlir_module_fingerprint = 0, property.num_replicas = 8, topology.chip_bounds().x = 2, topology.chip_bounds().y = 2, topology.chip_bounds().z = 1, topology.wrap().x = false, topology.wrap().y = false, topology.wrap().z = false, std::string(property.shapes_prefix) = "6,40,;6,206,;", property.guaranteed_constants_size = 0, embedding_partiti

2330/2330 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: nan - loss: nan

I0000 00:00:1745168970.815364      10 encapsulate_tpu_computations_pass.cc:266] Subgraph fingerprint:10941172077188220918
I0000 00:00:1745168971.110858     835 tpu_compilation_cache_interface.cc:442] TPU host compilation cache miss: cache_key(50796686282111353), session_name()
I0000 00:00:1745168971.334994     835 tpu_compile_op_common.cc:245] Compilation of 50796686282111353 with session name  took 224.073807ms and succeeded
I0000 00:00:1745168971.336985     835 tpu_compilation_cache_interface.cc:476] TPU host compilation cache: compilation complete for cache_key(50796686282111353), session_name(), subgraph_key(std::string(property.function_name) = "cluster_one_step_on_data_10941172077188220918", property.function_library_fingerprint = 814155877249253991, property.mlir_module_fingerprint = 0, property.num_replicas = 8, topology.chip_bounds().x = 2, topology.chip_bounds().y = 2, topology.chip_bounds().z = 1, topology.wrap().x = false, topology.wrap().y = false, topology.wrap().z = fals

2330/2330 ━━━━━━━━━━━━━━━━━━━━ 39s 14ms/step - accuracy: nan - loss: nan - val_accuracy: nan - val_loss: nan - learning_rate: 0.0010
Epoch 2/50
2330/2330 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - accuracy: nan - loss: nan - val_accuracy: nan - val_loss: nan - learning_rate: 0.0010
Epoch 3/50
2330/2330 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - accuracy: nan - loss: nan - val_accuracy: nan - val_loss: nan - learning_rate: 0.0010
Epoch 4/50
2330/2330 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - accuracy: nan - loss: nan - val_accuracy: nan - val_loss: nan - learning_rate: 2.0000e-04
Epoch 5/50
2330/2330 ━━━━━━━━━━━━━━━━━━━━ 29s 12ms/step - accuracy: nan - loss: nan - val_accuracy: nan - val_loss: nan - learning_rate: 2.0000e-04
Epoch 6/50
2330/2330 ━━━━━━━━━━━━━━━━━━━━ 30s 12ms/step - accuracy: nan - loss: nan - val_accuracy: nan - val_loss: nan - learning_rate: 2.0000e-04
